# Notebook 04 - Building a Multimodal Benchmark Harness

This notebook builds the measurement layer for multimodal work: datasets, prediction records, simple scorers, and aggregate reporting.


## What you will learn

- how to represent multimodal benchmark examples with explicit evidence metadata
- how to score extraction and grounded-answer tasks
- how to summarize results by modality and failure type


In [ ]:
!pip install -q numpy pandas matplotlib pydantic
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "04_building_a_multimodal_benchmark_harness"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Define benchmark records

A good benchmark row stores the task, modality, expected answer, and the evidence span that should support the answer.


In [ ]:
@dataclass
class BenchmarkExample:
    example_id: str
    modality: str
    question: str
    expected_answer: str
    expected_evidence: str

examples = [
    BenchmarkExample("img-001", "image", "What trend does the chart show?", "sales rise in Q4", "chart:bars:q4"),
    BenchmarkExample("doc-001", "document", "What is the invoice total?", "$1,240.00", "page:1:field:total"),
    BenchmarkExample("aud-001", "audio", "What does the speaker request?", "schedule a follow-up", "00:11-00:16"),
    BenchmarkExample("vid-001", "video", "What breaks during the demo?", "the login flow after password reset", "frame:18-26"),
]
pd.DataFrame([asdict(example) for example in examples])


## Step 2: Score predictions

Keep the first harness simple: exact answer checks, evidence overlap checks, and a combined grounded-success flag.


In [ ]:
def normalize(text):
    return " ".join(str(text).lower().strip().replace("$", "").split())

def exact_match(prediction, target):
    return int(normalize(prediction) == normalize(target))

predictions = pd.DataFrame([
    {"example_id": "img-001", "prediction": "sales rise in q4", "evidence": "chart:bars:q4"},
    {"example_id": "doc-001", "prediction": "$1,240.00", "evidence": "page:1:field:total"},
    {"example_id": "aud-001", "prediction": "schedule a follow up", "evidence": "00:10-00:17"},
    {"example_id": "vid-001", "prediction": "the login flow after password reset", "evidence": "frame:20-24"},
])
reference = pd.DataFrame([asdict(example) for example in examples])
scored = predictions.merge(reference, on="example_id")
scored["answer_score"] = scored.apply(lambda row: exact_match(row.prediction, row.expected_answer), axis=1)
scored["evidence_hit"] = (scored["evidence"].str.split(":").str[0] == scored["expected_evidence"].str.split(":").str[0]).astype(int)
scored["grounded_success"] = ((scored["answer_score"] == 1) & (scored["evidence_hit"] == 1)).astype(int)
scored[["example_id", "modality", "answer_score", "evidence_hit", "grounded_success"]]


## Step 3: Aggregate the report

Benchmark reports should quickly show where a system fails by modality instead of hiding everything inside one average.


In [ ]:
summary = scored.groupby("modality")[["answer_score", "evidence_hit", "grounded_success"]].mean().round(3)
summary.loc["overall"] = scored[["answer_score", "evidence_hit", "grounded_success"]].mean().round(3)
summary


## Exercises and extensions

1. Add a confidence column and build a calibration table by modality.
1. Extend the harness with a failure_bucket column so you can separate wrong answer, wrong evidence, and missing output cases.
